# End-to-End Cancellation Risk ML Workflow

This notebook is the heavy-compute training path for the Streamlit application. It can run in ordinary Google Colab with a T4 GPU or in Colab Enterprise.

It performs: **data ingestion → validation/versioning → feature preparation → five-model training → MLflow tracking → qualification gates → candidate selection → candidate export**.

The notebook never deploys a model automatically. The best qualified model is exported as a **Candidate** and must be promoted from the Streamlit Model Registry.

In [ ]:
# 1) Configuration
REPO_URL = "https://github.com/YOUR_ORG/YOUR_REPO.git"   # change once
BRANCH = "main"
DATASET_VERSION = "pakistan_seed_v1"

# Optional: point to a new raw item-level CSV.
# Ordinary Colab: a Google Drive path after Drive is mounted.
# Colab Enterprise: a local path staged by your runtime/startup process.
RAW_CSV_PATH = ""

# Ordinary Colab persistence. If Drive can be mounted, experiments are stored here.
DRIVE_ROOT = "/content/drive/MyDrive/cancellation-risk-mlops"

# Optional Colab Enterprise persistence. Set to gs://bucket/path when using an execution job.
# The notebook trains locally, then copies the experiment folder here at the end.
GCS_OUTPUT_ROOT = ""

USE_GPU = True
RANDOM_SEED = 42


## Runtime
For faster tree boosting, select **Runtime → Change runtime type → T4 GPU** in ordinary Colab. XGBoost and LightGBM will attempt GPU training and transparently fall back to CPU if the installed build does not support GPU. Random Forest and Extra Trees remain CPU models.

In [ ]:
# 2) Detect ordinary Colab vs Colab Enterprise and inspect the accelerator
from pathlib import Path
import os, subprocess

IS_ORDINARY_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_ORDINARY_COLAB = True
except Exception:
    print('Google Drive mount is unavailable; using local execution storage (appropriate for Colab Enterprise).')

if IS_ORDINARY_COLAB:
    PERSIST_ROOT = Path(DRIVE_ROOT)
else:
    PERSIST_ROOT = Path('/content/cancellation-risk-mlops')
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
except Exception:
    print('No GPU detected. The workflow will still run on CPU.')
print('Persistent/local experiment root:', PERSIST_ROOT)


In [ ]:
# 3) Clone/update the application repository
import os, subprocess, pathlib
PROJECT_DIR = pathlib.Path('/content/cancellation-risk-app')
if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)


In [ ]:
# 4) Install the reproducible training environment
%pip install -q -r requirements.txt

import sys, os
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)


## DataOps input
If `RAW_CSV_DRIVE_PATH` is supplied, the notebook runs the same ingestion/validation/versioning code used by the developer DataOps page. Otherwise it starts from the packaged, already validated Pakistan order-level seed dataset.

In [ ]:
# 5) Load or ingest data
from pathlib import Path
import pandas as pd

if RAW_CSV_PATH:
    from src.bootstrap import initialize_runtime
    from src.data.ingestion import ingest_batch
    from src.data.io import read_snapshot
    initialize_runtime()
    raw_path = Path(RAW_CSV_PATH)
    result = ingest_batch(raw_path.read_bytes(), raw_path.name, run_mode='full')
    if result.get('status') != 'success':
        raise RuntimeError(result)
    DATASET_VERSION = result['dataset_version']
    snapshot = read_snapshot(Path(result['processed_path']))
    print('Created dataset version:', DATASET_VERSION)
else:
    seed = PROJECT_DIR / 'data' / 'seed' / 'pakistan_orders_v1.csv.gz'
    snapshot = pd.read_csv(seed, parse_dates=['created_at'], low_memory=False)
    print('Using packaged dataset:', DATASET_VERSION, 'orders:', len(snapshot))


In [ ]:
# 6) Configure MLflow + experiment output
from datetime import datetime, timezone
from pathlib import Path

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
MLFLOW_DIR = PERSIST_ROOT / 'mlflow'
OUTPUT_DIR = PERSIST_ROOT / 'experiments' / DATASET_VERSION / stamp
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = MLFLOW_DIR.resolve().as_uri()
print('MLflow:', MLFLOW_TRACKING_URI)
print('Output:', OUTPUT_DIR)


## Train the five-model suite
All models use the same time-based train/validation/test split and the same leakage-safe feature contract. MLflow records parameters, metrics, business metrics, device, dataset version, random seed, Git commit, and model artifacts.

In [ ]:
# 7) Train + track + select Candidate
from src.models.suite import SuiteConfig, train_model_suite

def progress(stage, fraction, message):
    print(f'[{stage:10s}] {fraction:6.1%}  {message}')

config = SuiteConfig(
    dataset_version=DATASET_VERSION,
    output_dir=OUTPUT_DIR,
    use_gpu=USE_GPU,
    random_seed=RANDOM_SEED,
    mlflow_tracking_uri=MLFLOW_TRACKING_URI,
)
summary = train_model_suite(snapshot, config, progress=progress)
print('Candidate:', summary['candidate_model_id'])
print('Candidate package:', summary['candidate_package'])


In [ ]:
# 8) Compare results
import pandas as pd
rows = []
for m in summary['models']:
    rows.append({
        'Model': m['display_name'],
        'Device': m['training_device'],
        'Qualified': m.get('qualification', {}).get('passed'),
        'ROC-AUC': m['test_metrics']['roc_auc'],
        'PR-AUC': m['test_metrics']['pr_auc'],
        'Brier': m['test_metrics']['brier'],
        'Recall @ 90% precision': m['test_metrics'].get('recall_at_90_precision'),
        'Net savings / 1,000 (PKR)': m['business_metrics']['net_savings_per_1000_orders'],
        'Selection score': m.get('selection_score'),
    })
results = pd.DataFrame(rows).sort_values('Selection score', ascending=False, na_position='last')
display(results)


## Candidate handoff
The ZIP below contains the selected model and support artifacts. Download it and use **Developer → Model Registry & Deployment → Import candidate package**. Importing it does **not** deploy it. A developer must still click **Promote Candidate to Production**.

In [ ]:
# 9) Candidate handoff / persistence
from pathlib import Path
import subprocess
package = Path(summary['candidate_package']) if summary.get('candidate_package') else None

if GCS_OUTPUT_ROOT:
    target = GCS_OUTPUT_ROOT.rstrip('/') + f'/{DATASET_VERSION}/{stamp}/'
    print('Copying experiment output to:', target)
    subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(OUTPUT_DIR), target], check=True)

if package and package.exists():
    print('Candidate package:', package)
    if IS_ORDINARY_COLAB:
        from google.colab import files
        files.download(str(package))
    else:
        print('Colab Enterprise: retrieve this package from GCS_OUTPUT_ROOT after the copy step, or from the execution output artifact.')
else:
    print('No model passed the qualification gates. Review the experiment metrics and gates.')


## Reproducibility
The workflow records the dataset version, time-based split, random seed, package versions, Git commit SHA, model hyperparameters, training device, MLflow run IDs, and business assumptions. Re-running the same commit and data version with the same seed provides a reproducible experiment baseline, subject to normal hardware/library numerical variation.